# DWD 层 — 清洗统一 + 星型模型

ODS Yellow + Green + fhv → 统一字段口径 → 清洗 → 衍生列 → 写出 `fact_trips`

同时构建维度表：`dim_zone`、`dim_date`

## 1. SparkSession（启用动态分区覆盖）

In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType

spark = (
    SparkSession.builder
    .appName("TLC-DWD-Build")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.sources.partitionOverwriteMode", "dynamic")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

BASE            = os.path.expanduser("~/tlc-spark-pipeline")
ODS_Y           = f"{BASE}/data/warehouse/ods/yellow"
ODS_G           = f"{BASE}/data/warehouse/ods/green"
ODS_FHV         = f"{BASE}/data/warehouse/ods/fhv_hv"
DWD_FACT        = f"{BASE}/data/warehouse/dwd/fact_trips"
DWD_ZONE        = f"{BASE}/data/warehouse/dwd/dim_zone"
DWD_DATE        = f"{BASE}/data/warehouse/dwd/dim_date"
DWD_CLEANING    = f"{BASE}/data/warehouse/dq/dwd_cleaning"   # records dropped by DWD cleaning
ZONE_CSV        = f"{BASE}/data/raw/taxi_zone_lookup.csv"

TARGET_DATE = os.environ.get("TARGET_DATE", "2024-03-15")
yr  = int(TARGET_DATE.split("-")[0])
mo  = int(TARGET_DATE.split("-")[1])
day = int(TARGET_DATE.split("-")[2])
print(f"TARGET_DATE={TARGET_DATE}")
print(f"Spark {spark.version} ready | TARGET_DATE={TARGET_DATE}")


## 2. 统一字段口径

黄色多 `Airport_fee`，绿色多 `trip_type` / `ehail_fee`。统一为相同列集合，差异字段补 null。

In [ ]:
yellow = spark.read.parquet(ODS_Y).filter(
    (F.col("pickup_year") == yr) & (F.col("pickup_month") == mo) & (F.col("pickup_day") == day))
green  = spark.read.parquet(ODS_G).filter(
    (F.col("pickup_year") == yr) & (F.col("pickup_month") == mo) & (F.col("pickup_day") == day))
fhv    = spark.read.parquet(ODS_FHV).filter(
    (F.col("pickup_year") == yr) & (F.col("pickup_month") == mo) & (F.col("pickup_day") == day))

# 黄色：补绿色独有列 + FHV 独有列（null 占位，确保 common_cols 包含这些字段）
yellow_unified = (
    yellow
    .withColumn("trip_type",        F.lit(None).cast("int"))
    .withColumn("Airport_fee",      F.col("Airport_fee"))
    .withColumn("sales_tax",        F.lit(0.0).cast("double"))
    .withColumn("bcf",              F.lit(0.0).cast("double"))
    .withColumn("driver_pay",       F.lit(None).cast("double"))   # FHV-only, null for taxi
    .withColumn("request_datetime", F.lit(None).cast("timestamp")) # FHV-only, null for taxi
)

# 绿色：补黄色独有列 + FHV 独有列
green_unified = (
    green
    .withColumn("Airport_fee",      F.lit(0.0).cast("double"))
    .withColumn("sales_tax",        F.lit(0.0).cast("double"))
    .withColumn("bcf",              F.lit(0.0).cast("double"))
    .withColumn("driver_pay",       F.lit(None).cast("double"))
    .withColumn("request_datetime", F.lit(None).cast("timestamp"))
    .drop("ehail_fee")
)

# FHV HV：字段映射到统一口径；total_amount 包含 sales_tax + bcf（出租车无此税项）
# driver_pay 和 request_datetime 直接保留（FHV ODS 原有字段）
fhv_unified = (
    fhv
    .withColumnRenamed("base_passenger_fare", "fare_amount")
    .withColumnRenamed("trip_miles",          "trip_distance")
    .withColumnRenamed("tips",                "tip_amount")
    .withColumnRenamed("tolls",               "tolls_amount")
    .withColumnRenamed("airport_fee",         "Airport_fee")
    .withColumn("passenger_count",       F.lit(None).cast("int"))
    .withColumn("payment_type",          F.lit(None).cast("int"))
    .withColumn("improvement_surcharge", F.lit(0.0).cast("double"))
    .withColumn("mta_tax",               F.lit(0.0).cast("double"))
    .withColumn("trip_type",             F.lit(None).cast("int"))
    # total_amount for FHV HV: fare + tolls + congestion + airport + tips + sales_tax + bcf
    .withColumn("total_amount",
        F.coalesce(F.col("fare_amount"),          F.lit(0)) +
        F.coalesce(F.col("tolls_amount"),         F.lit(0)) +
        F.coalesce(F.col("congestion_surcharge"), F.lit(0)) +
        F.coalesce(F.col("Airport_fee"),          F.lit(0)) +
        F.coalesce(F.col("tip_amount"),           F.lit(0)) +
        F.coalesce(F.col("sales_tax"),            F.lit(0)) +
        F.coalesce(F.col("bcf"),                  F.lit(0)))
)

common_cols = [c for c in yellow_unified.columns
               if c in green_unified.columns and c in fhv_unified.columns]

combined = (
    yellow_unified.select(common_cols)
    .union(green_unified.select(common_cols))
    .union(fhv_unified.select(common_cols))
)

print(f"Combined rows (pre-clean): {combined.count():,}")
print(f"Columns ({len(common_cols)}): {common_cols}")


## 3. 清洗 + 衍生列

In [ ]:
from pyspark.sql.functions import col, unix_timestamp, round as spark_round, hour, month, year, when, dayofweek

# 1. 先计算衍生列（speed/duration 的清洗判断依赖时间差，须先算出）
with_derived = (
    combined
    .withColumn("trip_duration_min",
        spark_round(
            (unix_timestamp("dropoff_datetime") - unix_timestamp("pickup_datetime")) / 60, 2))
    .withColumn("avg_speed_mph",
        F.when(
            unix_timestamp("dropoff_datetime") > unix_timestamp("pickup_datetime"),
            spark_round(col("trip_distance") /
                ((unix_timestamp("dropoff_datetime") - unix_timestamp("pickup_datetime")) / 3600), 2)
        ).otherwise(F.lit(None))
    )
)

# 2. 标注违规原因（结构性错误才隔离；已知系统性时钟漂移在 fact 上打标，不隔离）
marked = (
    with_derived
    .withColumn("dq_reason",
        F.when(col("fare_amount") <= 0,            "non_positive_fare")
        .when(col("trip_distance") <= 0,            "non_positive_trip_distance")
        .when(col("dropoff_datetime") <= col("pickup_datetime"), "invalid_time_order")
        .when(col("PULocationID").isNull(),          "null_PULocationID")
        .when(col("DOLocationID").isNull(),          "null_DOLocationID")
        .when(col("passenger_count").isNotNull() &
              ~col("passenger_count").between(1, 6), "invalid_passenger_count")
        .when((col("pickup_datetime") < "2024-01-01") |
              (col("pickup_datetime") >= "2024-04-01"), "out_of_range_datetime")
        .when(col("avg_speed_mph").isNotNull() &
              (col("avg_speed_mph") > 100),          "excessive_speed")
        .when(~col("trip_duration_min").between(1, 180), "invalid_duration")
        # request_after_pickup 不再隔离：调度平台时钟漂移每天稳定 ~1%，
        # 车费/地点/时长完整 → 改为 fact 上打标（同 passenger_count_missing）
    )
)

# 3. 分流：干净记录 → fact_trips；违规记录 → DWD 清洗隔离表
rejected = (
    marked
    .filter(col("dq_reason").isNotNull())
    .select("source_type", "pickup_year", "pickup_month", "pickup_day",
            "pickup_datetime", "dropoff_datetime", "PULocationID", "DOLocationID",
            "fare_amount", "trip_distance", "passenger_count",
            "trip_duration_min", "avg_speed_mph", "dq_reason")
    .withColumn("check_ts", F.current_timestamp())
)

fact = (
    marked
    .filter(col("dq_reason").isNull())
    .drop("dq_reason")
    # passenger_count_missing: null 记录保留在 fact，打标（不隔离）
    .withColumn("passenger_count_missing", col("passenger_count").isNull())
    # request_after_pickup: FHV HV 调度平台时钟漂移，打标不隔离（每天稳定 ~1%）
    .withColumn("request_after_pickup",
        (col("source_type") == "fhv_hv") &
        col("request_datetime").isNotNull() &
        (col("request_datetime") > col("pickup_datetime")))
    .withColumn("fare_per_mile",
        spark_round(col("fare_amount") / col("trip_distance"), 2))
    .withColumn("surcharge_total",
        spark_round(
            F.coalesce(col("improvement_surcharge"), F.lit(0)) +
            F.coalesce(col("mta_tax"), F.lit(0)) +
            F.coalesce(col("congestion_surcharge"), F.lit(0)) +
            F.coalesce(col("Airport_fee"), F.lit(0)), 2))
    .withColumn("time_of_day",
        when(hour("pickup_datetime").between(6,  9),  "morning_rush")
        .when(hour("pickup_datetime").between(10, 15), "midday")
        .when(hour("pickup_datetime").between(16, 19), "evening_rush")
        .when(hour("pickup_datetime").between(20, 23), "night")
        .otherwise("late_night"))
    .withColumn("payment_label",
        when(col("payment_type") == 1, "credit_card")
        .when(col("payment_type") == 2, "cash")
        .when(col("payment_type") == 3, "no_charge")
        .otherwise("other"))
    .withColumn("pickup_year",  year("pickup_datetime"))
    .withColumn("pickup_month", month("pickup_datetime"))
    .withColumn("pickup_hour",  hour("pickup_datetime"))
    .withColumn("day_of_week",  dayofweek("pickup_datetime"))
)

fact_count     = fact.count()
rejected_count = rejected.count()
combined_count = fact_count + rejected_count
print(f"Combined (pre-clean): {combined.count():,}")
print(f"fact_trips:           {fact_count:,}")
print(f"rejected (cleaning):  {rejected_count:,}")
print(f"Conservation check:   {combined.count() == combined_count}  ({combined_count:,} = {fact_count:,} + {rejected_count:,})")
fact.groupBy("source_type", "pickup_month").count().orderBy("source_type", "pickup_month").show()
print("\nRejected breakdown:")
rejected.groupBy("source_type", "dq_reason").count().orderBy("source_type", F.desc("count")).show()
print(f"\nrequest_after_pickup tag: {fact.filter(col('request_after_pickup')).count():,} records")

## 4. 写出 fact_trips（动态分区覆盖）

In [ ]:
# Write fact_trips (dynamic partition overwrite — only touches TARGET_DATE partition)
(
    fact
    .write
    .partitionBy("source_type", "pickup_year", "pickup_month", "pickup_day")
    .mode("overwrite")
    .parquet(DWD_FACT)
)
print(f"fact_trips written to: {DWD_FACT}")

# Write DWD cleaning quarantine (rejected records with dq_reason)
# Partitioned by source_type + date so each daily run overwrites only its partition
(
    rejected
    .write
    .partitionBy("source_type", "pickup_year", "pickup_month", "pickup_day")
    .mode("overwrite")
    .parquet(DWD_CLEANING)
)
print(f"DWD cleaning quarantine written to: {DWD_CLEANING}  ({rejected_count:,} records)")


## 5. 维度表：dim_zone

In [ ]:
# Build dim_zone from TLC official taxi zone lookup CSV (265 zones)
# Includes zone name and borough — far more useful than just location_id
dim_zone = (
    spark.read.csv(ZONE_CSV, header=True, inferSchema=True)
    .withColumnRenamed("LocationID",   "location_id")
    .withColumnRenamed("Borough",      "borough")
    .withColumnRenamed("Zone",         "zone_name")
    .withColumnRenamed("service_zone", "service_zone")
)
dim_zone.write.mode("overwrite").parquet(DWD_ZONE)
print(f"dim_zone: {dim_zone.count()} zones → {DWD_ZONE}")
dim_zone.show(5)


## 6. 维度表：dim_date

In [ ]:
from datetime import date, timedelta
from pyspark.sql.types import DateType

# Generate all Q1 2024 dates once — not derived from fact_trips (which changes daily)
# This is idempotent: running daily always produces the same full Q1 dim_date
q1_dates = [(date(2024, 1, 1) + timedelta(days=i),) for i in range(91)]  # Jan 1 – Mar 31

dim_date = (
    spark.createDataFrame(q1_dates, ["date"])
    .withColumn("year",        F.year("date"))
    .withColumn("month",       F.month("date"))
    .withColumn("day",         F.dayofmonth("date"))
    .withColumn("day_of_week", F.dayofweek("date"))
    .withColumn("week",        F.weekofyear("date"))
    .withColumn("quarter",     F.quarter("date"))
    .withColumn("is_weekend",  F.dayofweek("date").isin(1, 7).cast("boolean"))
    .orderBy("date")
)

dim_date.write.mode("overwrite").parquet(DWD_DATE)
print(f"dim_date: {dim_date.count()} dates (2024-01-01 to 2024-03-31) → {DWD_DATE}")
dim_date.show(5)

spark.stop()
print("DWD build complete.")
